In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# IBM Circuit function

*[API 참조](https://docs.quantum.ibm.com/api/functions/ibm-circuit-function)를 참조하세요*

> **Note:** * Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 현재 미리 보기 릴리스 상태이며 변경될 수 있습니다.

## 개요
IBM&reg; Circuit function은 [추상 PUB](/guides/primitive-input-output#pubs)을 입력으로 받아, 완화된 기댓값을 출력으로 반환합니다. 이 Circuit function에는 연구자들이 알고리즘 및 애플리케이션 탐구에 집중할 수 있도록 자동화되고 맞춤화된 파이프라인이 포함되어 있습니다.

## 설명
PUB를 제출하면, 추상 Circuit과 관측값이 자동으로 트랜스파일되고, 하드웨어에서 실행되며, 완화된 기댓값을 반환하기 위해 후처리됩니다. 이를 위해 다음 도구들을 결합합니다.

- [Qiskit Transpiler Service](/guides/qiskit-transpiler-service): AI 기반 및 휴리스틱 트랜스파일 패스의 자동 선택을 통해 추상 Circuit을 하드웨어 최적화된 ISA Circuit으로 변환합니다.
- [유틸리티 규모 계산에 필요한 오류 억제 및 완화](/guides/error-mitigation-and-suppression-techniques): 측정 및 Gate 트위를링, 동적 디커플링, Twirled Readout Error eXtinction(TREX), Zero-Noise Extrapolation(ZNE), Probabilistic Error Amplification(PEA)이 포함됩니다.
- [Qiskit Runtime Estimator](/guides/get-started-with-estimator): ISA PUB를 하드웨어에서 실행하고 완화된 기댓값을 반환합니다.

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## 시작하기
[API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고 다음과 같이 Qiskit Function을 선택하세요. (이 코드 스니펫은 이미 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)해 두었다고 가정합니다.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## 예제
시작하려면 다음 기본 예제를 사용해 보세요.

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Qiskit Function 워크로드의 [상태](/guides/functions#check-job-status)를 확인하거나 다음과 같이 [결과](/guides/functions#retrieve-results)를 반환하세요.

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


결과는 [Estimator 결과](/guides/estimator-input-output)와 동일한 형식입니다.

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

### 완화 수준 예제
다음 예제는 완화 수준 설정 방법을 보여줍니다.

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


다음 예제에서는 완화 수준을 1로 설정하면 초기에 ZNE 완화가 꺼지지만, `zne_mitigation`을 `True`로 설정하면 `mitigation_level`의 관련 설정이 재정의됩니다.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

### 출력 예제
다음 코드 스니펫은 `PrimitiveResult`(및 관련 `PubResult`) 형식을 설명합니다.